# Cleaning: KZN Local Government Elections 2000 and 2006

**Added to the pipeline after the original 2011-2021 cleaning and modeling stages were already complete.**

While reviewing the modeling results for the 2011-2021 panel, we located two additional
national Local Government Election result files (2000 and 2006) covering all nine provinces.
We filtered both to KwaZulu-Natal only and are cleaning them here, following the same four
checks used throughout this project: missing values, inconsistent formats, wrong data types,
and duplicates/junk rows.

Source files (national, all provinces): `2000_LGE.csv`, `2006_LGE.csv`
Already filtered to KZN only before this notebook: `KZN_2000_raw_filtered.csv`, `KZN_2006_raw_filtered.csv`


In [ ]:
import pandas as pd
import re
import os

# Make sure we're working from the repo root so relative paths behave consistently
# (adjust the number of ".." if this notebook sits at a different depth)
# os.chdir("../..")

print("Working directory:", os.getcwd())


## Step 1: Load the KZN-filtered raw files

These were filtered from the national files down to `Province == "KwaZulu-Natal"` only,
matching the province scope of the rest of this project.


In [ ]:
df2000_raw = pd.read_csv("data/raw/KZN_Local_Government_Elections_2000-2006/KZN_2000_LGE_raw_filtered.csv")
df2006_raw = pd.read_csv("data/raw/KZN_Local_Government_Elections_2000-2006/KZN_2006_LGE_raw_filtered.csv")

print("2000 raw shape:", df2000_raw.shape)
print("2006 raw shape:", df2006_raw.shape)
print("\n2000 columns:", df2000_raw.columns.tolist())
print("2006 columns:", df2006_raw.columns.tolist())


## Step 2: Fix inconsistent formats and wrong data types

Column names contain literal line breaks (e.g. `"Registered\nVoters"`). `RegisteredVoters` is
stored as text with thousands separators (`"1,721"`). The turnout percentage is stored as text
with a `%` sign. We fix all three here, and standardise column names to match the style already
used in the 2011/2016/2021 cleaned files (`RegisteredVoters`, `VotingDistrict`, `TotalVotesCast`,
`ValidVotesCast`, `SpoiltVotes`).


In [ ]:
def clean_lge(df, year):
    df = df.copy()
    df.columns = [re.sub(r"\s+", " ", c).strip() for c in df.columns]
    df = df.rename(columns={
        "Voting District": "VotingDistrict",
        "Ballot Type": "BallotType",
        "Registered Voters": "RegisteredVoters",
        "% Voter Turnout": "PercentTurnoutReported",
        "Total Votes Cast": "TotalVotesCast",
        "Valid Votes Cast": "ValidVotesCast",
        "Spoilt Votes": "SpoiltVotes",
    })
    df["RegisteredVoters"] = df["RegisteredVoters"].astype(str).str.replace(",", "").astype(float)
    df["PercentTurnoutReported"] = df["PercentTurnoutReported"].astype(str).str.replace("%", "").astype(float)
    df["TotalVotesCast"] = pd.to_numeric(df["TotalVotesCast"], errors="coerce")
    df["ValidVotesCast"] = pd.to_numeric(df["ValidVotesCast"], errors="coerce")
    df["Ward"] = df["Ward"].astype(float).astype("Int64")
    df["ElectionYear"] = year

    keep = ["Province", "Municipality", "Ward", "VotingDistrict", "BallotType",
            "RegisteredVoters", "TotalVotesCast", "ValidVotesCast", "ElectionYear"]
    return df[keep]

df2000_clean = clean_lge(df2000_raw, 2000)
df2006_clean = clean_lge(df2006_raw, 2006)

print("2000 cleaned shape:", df2000_clean.shape)
print("2006 cleaned shape:", df2006_clean.shape)
df2000_clean.head()


## Step 3: Duplicates and junk row check


In [ ]:
for name, df in [("2000", df2000_clean), ("2006", df2006_clean)]:
    dupes = df.duplicated().sum()
    nulls = df.isna().sum().sum()
    print(f"{name}: {dupes} exact duplicate rows, {nulls} total missing values across all columns")


## Step 4: Missing values check, column by column


In [ ]:
print("2000 missing values by column:")
print(df2000_clean.isna().sum())
print("\n2006 missing values by column:")
print(df2006_clean.isna().sum())


## Step 5: Save cleaned files to interim

These are saved as a new interim subfolder, since the existing
`KZN_Local_Goverment_elections_2011-21 Cleaned/` folder covers a different, already-finished
year range and should not be altered.


In [ ]:
output_dir = "data/interim/KZN_Local_Government_Elections_2000-2006 Cleaned/"
os.makedirs(output_dir, exist_ok=True)

df2000_clean.to_csv(output_dir + "KZN_2000_clean.csv", index=False)
df2006_clean.to_csv(output_dir + "KZN_2006_clean.csv", index=False)

print("Saved:")
print(output_dir + "KZN_2000_clean.csv", "-", df2000_clean.shape)
print(output_dir + "KZN_2006_clean.csv", "-", df2006_clean.shape)


## Note on timing

This cleaning step was completed after the original 2011-2021 election data had already been
cleaned, merged, and used to train the first version of the model. These two files were found
afterwards and are being brought in through the same pipeline stages, in order, rather than
inserted directly into the finished model. See `Why_2000_2006_Were_Added_Later.docx` for the
full reasoning and what this changed.
